## Environment Setup & Imports, as well as resume data ingestion

In [ ]:
import os
import json
from dotenv import load_dotenv

# Load API keys from .env file
load_dotenv()

# Import matching agent graph and standalone tools
from src.agent.matching_agent import (
    build_matching_agent,
    compare_candidates,
    generate_interview_questions
)
from src.tools.fs_tools import list_files, read_file

print("✅ Setup complete and modules imported successfully!")


# Ingest sample resumes from data/resumes directory
resume_files = list_files("data/resumes")
candidate_pool = []

for idx, file_path in enumerate(resume_files):
    content = read_file(file_path)
    candidate_name = os.path.basename(file_path).replace(".pdf", "").replace(".txt", "").replace("_", " ").title()
    candidate_pool.append({
        "id": f"cand_{idx+1}",
        "name": candidate_name,
        "file_path": file_path,
        "content": content
    })

print(f"✅ Ingested {len(candidate_pool)} resumes into candidate pool.")

## Scenario 1 — Basic JD Parsing & Retrieval

In [ ]:
# Initialize compiled agent
agent = build_matching_agent()

sample_jd = """
Senior Full Stack Engineer
Must Have:
- 3+ years experience with Python and React
- Knowledge of vector databases and RAG workflows
Nice to Have:
- Experience with LangGraph and Streamlit
"""

initial_state = {
    "messages": [],
    "raw_jd": sample_jd,
    "must_have_reqs": [],
    "nice_have_reqs": [],
    "candidate_pool": candidate_pool,
    "shortlist": [],
    "reasoning": {},
    "report": "",
    "human_feedback": None,
    "screening_round": 1
}

output_s1 = agent.invoke(initial_state)

print("=== Scenario 1 Output ===")
print(f"Extracted Must-Haves: {output_s1['must_have_reqs']}")
print(f"Shortlisted Candidates Count: {len(output_s1['shortlist'])}")

## Scenario 2 — Mid-Conversation Criteria Refinement

In [ ]:
# Add user refinement feedback
output_s1["human_feedback"] = "Must have strong experience with PostgreSQL"

output_s2 = agent.invoke(output_s1)

print("=== Scenario 2 Output (After Criteria Refinement) ===")
print("Updated Messages History:")
for msg in output_s2["messages"][-2:]:
    print(f"- [{msg['role']}]: {msg['content']}")

## Head to Toe Candidate Comparison

In [ ]:
if len(candidate_pool) >= 2:
    cand_ids = [candidate_pool[0]["name"], candidate_pool[1]["name"]]
    comparison_report = compare_candidates(cand_ids, candidate_pool)
    
    print("=== Scenario 3 Output (Candidate Comparison) ===")
    print(comparison_report)
else:
    print("⚠️ At least 2 candidates required in candidate_pool for Scenario 3.")

## Scenario 4 — Interview Question Generation

In [ ]:
if candidate_pool:
    cand_id = candidate_pool[0]["name"]
    reqs = output_s1.get("must_have_reqs", ["Python", "React"])
    
    questions = generate_interview_questions(cand_id, candidate_pool, reqs)
    
    print("=== Scenario 4 Output (Tailored Interview Questions) ===")
    print(questions)

## Scenario 5 — Multi-Round Screening & Final Report

# Simulate Round 2 Deep Analysis
round2_state = output_s2.copy()
round2_state["screening_round"] = 2
round2_state["human_feedback"] = "approved"  # Complete feedback loop

output_s5 = agent.invoke(round2_state)

print("=== Scenario 5 Output (Final Evaluation Report) ===")
print(output_s5["report"])